# 📝 과제 01: 의약품 상담 도우미의 임베딩 모델 고르기

## 만들 결과

약국 체인 본사 검색 서버에 들어갈 **의약품 상담 도우미**의 임베딩 모델을 고릅니다. 식약처 「e약은요」 안내문 394편과 질문 24개로 BM25 기준선과 임베딩 후보 넷의 정확도를 측정하고, 선정 기준으로 모델 하나를 고릅니다.

- 후보 사양표를 만들고, 같은 질문으로 후보 넷을 임베딩합니다.
- 모델 비교표를 만들고, 선정 기준을 함수로 옮겨 모델을 고릅니다.
- 결정과 후보별 값을 `output/gwaje01_embedding_choice.json`에 남깁니다. 과제 02가 이 파일을 읽습니다.
- 결정을 ADR-001(임베딩 모델)로 남깁니다. 벡터를 어떤 정밀도로 저장할지는 저장소와 함께 과제 02에서 정합니다.

교안 01의 후보 비교를 의약품 안내문에 적용합니다. 지난 GraphRAG 변형 비교 단원의 과제는 안내문 18편에 GraphRAG·LightRAG를 돌려 근거를 넘겼는지 봤고, 이번 과제는 394편에서 모델·저장소·전략을 차례로 고르는 결정 과제입니다.

의학적 조언이 아니라 수업용 검색 자료이며, 실제 복용은 의사·약사와 상의해야 합니다.

**선정 기준**

서비스 가정:
- 약국 체인 본사의 검색 서버 하나가 매장 약사의 검색을 처리합니다.
- 질문에는 증상과 제품 이름만 들어가므로 외부 API를 써도 됩니다.
- 주의·상호작용은 안내문 뒤쪽에 있어 끝까지 읽어야 합니다.

우선순위(후보 모델마다 적용):

0. BM25는 기준선이라 고르지 않습니다.
1. 잘림 비율이 0인 모델만 남깁니다.
2. 맞힌 문항이 가장 많은 모델과 1문항 이내 차이는 같은 수준으로 봅니다(1문항 = Recall@5 약 4.2%p).
3. 같은 수준이면 차원이 작은 모델을 고릅니다(벡터가 작아 저장과 검색 부담이 적음).

교안과 다른 점:
- 교안 본실습은 정확도(2문항 이내), 잘림 1% 이하, 로컬 여부, 질문 시간 순으로 골랐습니다.
- 교안 따라하기(노무)는 외부 API 금지와 잘림 1% 이하로 먼저 거른 뒤 맞힌 문항이 가장 많은 모델을 골랐습니다.
- 과제는 외부 API를 허용하는 대신 잘림을 0으로 먼저 거르고, 같은 수준이면 차원이 작은 모델을 고릅니다. 고른 모델은 교안과 같을 수도 있습니다.

<img src="images/gwaje01_overview.png" width="1050" alt="임베딩 모델 고르기 흐름: 안내문 394편과 질문 24개를 BM25 기준선과 후보 넷 3-small (1536), 3-small (512), me5-small (384), KURE-v1 (1024)에 넣고, 후보 넷을 깔때기의 세 관문 1 잘림 0, 2 1문항 이내, 3 차원이 작은 쪽으로 거른다. 관문마다 빈 회색 카드가 빠져나가고, 출구에는 물음표 카드와 ADR-001 문서가 있다">

## 준비: 데이터와 제공 코드

- **자료:** `data/과제/`의 의약품 안내문 394편(`docs.jsonl`)과 질문 60개(`questions.json`) 가운데 묘사 12·이름 12개를 씁니다. 묘사 질문은 근거가 안내문 앞(효능·용법)에 있는 6개와 뒤(주의·상호작용·부작용)에 있는 6개입니다. 출처와 정리 규칙은 [자료 설명](data/과제/README.md)에 있습니다.
- **순서:** 리더보드 읽기 -> 사양표 -> 3-small 임베딩 -> 로컬 모델 임베딩 -> 비교표 -> 선정 함수 -> 결과 파일 -> ADR-001
- **제공:** 모델 내려받기, 환경 설정, 자료 읽기, 로컬 모델 불러오기, `normalize`, BM25 기준선, 평가·잘림 함수, 선정 보조 함수, `save_json`
- **작성:** `[작성]` 셀의 `...`를 번호 주석대로 채우고 바로 아래 자가채점 셀을 실행하세요. 3-small 셀만 OpenAI API를 부릅니다(문서 두 번에 약 80만 토큰, 약 $0.016).
- **시간:** 맥에서 노트북 전체가 약 2분 걸립니다. GPU가 없는 Windows PC에서는 KURE-v1 문서 임베딩만 3~6분 걸립니다.

In [ ]:
from huggingface_hub import snapshot_download

# 로컬 임베딩 모델 두 개(합쳐 약 2.8GB) 미리 내려받기 (교안 01에서 받았으면 캐시를 그대로 씀)
for model_name in ["nlpai-lab/KURE-v1", "intfloat/multilingual-e5-small"]:
    # 쓰지 않는 형식(onnx, openvino, pytorch_model.bin)은 제외
    snapshot_download(model_name, ignore_patterns=["onnx/*", "openvino/*", "pytorch_model.bin"])
    print(model_name, "준비 완료")

In [ ]:
# 라이브러리 불러오기
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

# data와 output 폴더가 있는 단원 폴더 기준 경로
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# API 키 읽기: 단원 .env 먼저, 없는 공통 키는 저장소 최상위 .env에서 (이미 읽은 키는 덮어쓰지 않음)
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )

In [ ]:
# 안내문과 질문 읽기 (docs.jsonl은 한 줄에 안내문 하나라 줄마다 읽음)
drugs = [json.loads(line) for line in (data_dir / "과제" / "docs.jsonl").read_text(encoding="utf-8").splitlines()]
questions = read_json("과제/questions.json")

# 원문 ID와 임베딩 입력 (제목 + 줄바꿈 + 본문, 모든 후보가 같은 입력)
drug_ids = [drug["doc_id"] for drug in drugs]
drug_texts = [drug["title"] + "\n" + drug["text"] for drug in drugs]

# 검색 한 번으로 답할 수 있는 묘사·이름 질문만 평가셋으로
eval_questions = []
for question in questions:
    if question["question_type"] in ("묘사", "이름"):
        eval_questions.append(question)
eval_texts = [question["question"] for question in eval_questions]

print("안내문:", len(drugs), "편 | 평가셋:", len(eval_questions), "문항 | 1문항 =", round(100 / len(eval_questions), 1), "%p")  # 1문항이 Recall@5를 몇 %p 움직이는지

# 평가셋 구성: 묘사는 근거 위치(앞: 효능·용법, 뒤: 주의·상호작용·부작용)로 나눔, 이름은 위치 없음
display(pd.DataFrame(eval_questions).value_counts(["question_type", "section"], dropna=False).rename("질문 수").to_frame())

## 1. 리더보드에서 공공 보건 데이터셋(PublicHealthQA) 점수를 읽으세요

리더보드는 검색 태스크 평균(`Retr.`)뿐 아니라 데이터셋마다의 점수도 보여 줍니다(`PERFORMANCE PER TASK` 탭). 교안은 평균으로 후보를 좁혔습니다. 의약품 안내문은 공공 보건 안내와 내용이 비슷하므로, 공공 보건 데이터셋(PublicHealthQA) 하나의 점수가 평균과 같은 후보 순서를 주는지 확인합니다.

**PublicHealthQA란?**
- MTEB(kor, v2)의 검색 태스크 하나로, 미국 질병통제예방센터(CDC)와 세계보건기구(WHO)의 자주 묻는 질문을 모은 공공 보건 질의응답 데이터셋입니다.
- 한국어를 포함한 여러 언어판이 있고, 한국어 벤치마크에는 한국어판 점수가 들어갑니다.

**작성:** 아래 기록 표와 두 문장(바로 아래 마크다운 셀)

**목적:** 평균 점수와 PublicHealthQA 점수로 매긴 후보 순서를 비교하고, 리더보드만으로 정할 수 없는 부분을 우리 평가로 넘깁니다.

**구현할 처리**
1. MTEB(kor, v2) 페이지의 `SUMMARY` 탭에서 KURE-v1과 me5-small 행의 `Retr.` 점수를 적습니다. OpenAI 3-small 행은 미리 채워 두었습니다.
2. 같은 페이지의 `PERFORMANCE PER TASK` 탭에서 KURE-v1과 me5-small 행의 `PublicHealthQA` 열 값을 적습니다. 표 위 검색창에 모델 이름(예: `KURE-v1`)을 넣으면 그 행만 남고, 표를 오른쪽으로 밀면 `PublicHealthQA` 열이 보입니다.
3. 점수가 없으면 `-`로 적고, 점수가 있는 열로만 순서를 비교합니다. 3-small은 한국어 검색 태스크 9개 가운데 일부만 평가되어 `Retr.`가 비어 있습니다. 같은 데이터셋·같은 지표(nDCG@10)로 세 후보를 모두 비교할 수 있는 것은 `PublicHealthQA` 열입니다.
4. 두 열의 후보 순서를 비교한 결과(같은지 다른지, 점수가 없는 후보는 어떻게 되는지)를 한 문장, 이 점수로 우리 안내문의 결론을 대신할 수 없는 이유를 한 문장 씁니다.

- 주소: <https://leaderboard.mteb.org/benchmark/MTEB%28kor%2C%20v2%29>
- 누를 곳: 표 바로 위 탭 줄의 `SUMMARY`(`Retr.` 열), 같은 줄 오른쪽의 `PERFORMANCE PER TASK`(검색창에 모델 이름을 넣고 표를 오른쪽으로 밀어 `PublicHealthQA` 열)

`SUMMARY` 탭: `Retr.` 열(검색 태스크 평균)과 모델 행

<img src="images/hf_mteb_retr_sort.png" width="1050" alt="MTEB(kor, v2) SUMMARY 탭. Retr. 열 머리글과 nlpai-lab/KURE-v1 행에 빨간 테두리">

`PERFORMANCE PER TASK` 탭: 검색창에 `KURE-v1`을 넣어 그 행만 남기고, 표를 오른쪽으로 밀어 `PublicHealthQA` 열의 값을 읽습니다.

<img src="images/hf_mteb_task_publichealthqa.png" width="1050" alt="MTEB(kor, v2) PERFORMANCE PER TASK 탭. 탭 이름, KURE-v1을 넣은 검색창, PublicHealthQA 열 머리글, KURE-v1 행과 그 행의 PublicHealthQA 값에 빨간 테두리">

| 모델 | `Retr.` | `PublicHealthQA` |
|---|---|---|
| nlpai-lab/KURE-v1 | | |
| intfloat/multilingual-e5-small | | |
| openai/text-embedding-3-small | - | 70.65 |

3-small 행의 `70.65`는 같은 탭에서 검색창에 `text-embedding-3`을 넣어 읽은 값입니다(아래 그림). OpenAI가 발표한 MIRACL 평균(3-small 44.0)은 다국어 검색 벤치마크라 데이터셋이 달라 이 표에 넣지 않습니다.

<img src="images/hf_mteb_task_openai.png" width="1050" alt="MTEB(kor, v2) PERFORMANCE PER TASK 탭. text-embedding-3을 넣은 검색창, PublicHealthQA 열 머리글, openai/text-embedding-3-large(85.31)와 openai/text-embedding-3-small(70.65) 행에 빨간 테두리">

**반환값:** 채운 기록 표와 두 문장. 리더보드는 새 결과가 올라오면 값이 바뀔 수 있으니 자기가 본 값을 적습니다.

*(여기에 기록 표를 옮겨 채우고, 두 열의 순서 비교와 리더보드만으로 정할 수 없는 이유를 한 문장씩 작성하세요.)*

사양표의 최대 입력은 로컬 모델 객체에서 읽으므로 두 모델을 먼저 불러옵니다.

In [ ]:
from sentence_transformers import SentenceTransformer

# 로컬 모델 두 개 불러오기 (맨 위에서 받은 캐시를 읽음)
me5_model = SentenceTransformer("intfloat/multilingual-e5-small")
kure_model = SentenceTransformer("nlpai-lab/KURE-v1")

# 실행 장치: 맥은 mps, NVIDIA GPU는 cuda, 없으면 cpu
print("me5-small:", me5_model.device, "| KURE-v1:", kure_model.device)

## 2. 후보 사양표를 만드세요

잘림 비율, 비용, 입력 규약은 모델마다 최대 입력·단가·접두어가 달라서 생기므로, 뒤의 임베딩과 계산이 모두 이 표를 읽습니다.

**작성:** 사양표 `model_specs`

**목적:** 교안 후보 사양표의 값 가운데 계산에 쓰는 여섯 가지를 후보 넷에 대해 dict로 옮깁니다.

**구현할 처리**
1. 키는 후보 이름 `"3-small (1536)"`, `"3-small (512)"`, `"me5-small (384)"`, `"KURE-v1 (1024)"` 네 개를 이 순서로 씁니다. BM25는 넣지 않습니다.
2. 값은 아래 여섯 키를 가진 dict입니다. 값은 교안 01 3-2 후보 사양표의 해당 행과 모델 카드에서 찾습니다(교안 표의 3-large 행은 이 과제 후보가 아닙니다).
3. 3-small (512)는 같은 3-small을 `dimensions`로 줄여 받는 설정이라 차원 말고는 3-small (1536)과 같습니다.

| 키 | 넣을 값 |
|---|---|
| `dimension` | 벡터 차원(정수). 후보 이름 괄호 안의 수 |
| `max_input` | 최대 입력 토큰 수(정수). OpenAI는 공식 문서 값, 로컬 모델은 모델 객체의 `max_seq_length` |
| `price_per_million` | 100만 토큰당 API 단가(달러, 실수). 로컬 모델은 `0.0` |
| `external_api` | 질문을 회사 밖 API로 보내면 `True`, 아니면 `False` |
| `query_prefix` | 질문 앞에 붙일 접두어. 없으면 `""`. me5-small은 `"query: "`처럼 콜론 뒤 공백까지 넣습니다 |
| `doc_prefix` | 문서 앞에 붙일 접두어. 없으면 `""`. me5-small은 `"passage: "`처럼 콜론 뒤 공백까지 넣습니다 |

**반환값:** `model_specs`(후보 이름을 키, 여섯 키 dict를 값으로 하는 dict)

In [ ]:
# [작성] 후보 넷의 사양표를 만드세요.
# 1. 키는 후보 이름, 값은 dimension·max_input·price_per_million·external_api·query_prefix·doc_prefix 여섯 키의 dict
# 2. 로컬 모델의 max_input은 me5_model, kure_model 객체에서 읽으세요.
model_specs = {
    "3-small (1536)": ...,
    "3-small (512)": ...,
    "me5-small (384)": ...,
    "KURE-v1 (1024)": ...,
}

In [ ]:
# [자가채점] 사양표의 키·자료형과 모델 객체 값을 확인합니다. 교안 사양표·모델 카드에서 옮긴 값은 답이 보이지 않게 지문(sha256 앞 10자리)으로 대조합니다.
import hashlib


def fingerprint(value):
    """값의 repr을 sha256으로 바꾼 앞 10자리를 반환합니다."""
    return hashlib.sha256(repr(value).encode("utf-8")).hexdigest()[:10]


candidate_names = ["3-small (1536)", "3-small (512)", "me5-small (384)", "KURE-v1 (1024)"]
spec_keys = {"dimension", "max_input", "price_per_million", "external_api", "query_prefix", "doc_prefix"}
prefix_fingerprints = {'3-small (1536)': '23c91fc33f', '3-small (512)': '23c91fc33f', 'me5-small (384)': '947a88145d', 'KURE-v1 (1024)': '23c91fc33f'}
local_models_check = {"me5-small (384)": me5_model, "KURE-v1 (1024)": kure_model}

assert isinstance(model_specs, dict), "model_specs에 dict를 담으세요. `...`를 채웠는지도 확인하세요."
assert list(model_specs) == candidate_names, "키는 후보 이름 네 개를 지문 순서대로 적으세요. BM25는 넣지 않습니다."
for name in candidate_names:
    spec = model_specs[name]
    assert isinstance(spec, dict) and set(spec) == spec_keys, f"{name}: 값은 지문 표의 여섯 키를 가진 dict입니다."
    # 후보 이름 괄호 안의 수가 차원
    assert spec["dimension"] == int(name.split("(")[1].rstrip(")")), f"{name}: dimension은 후보 이름 괄호 안의 수(정수)입니다."
    assert isinstance(spec["max_input"], int), f"{name}: max_input은 정수입니다."
    # 단가가 있는 API 모델만 질문을 회사 밖으로 보냄
    assert spec["external_api"] == (spec["price_per_million"] > 0), f"{name}: 단가가 있는 API 모델만 external_api가 True입니다."
    prefixes = (spec["query_prefix"], spec["doc_prefix"])
    assert fingerprint(prefixes) == prefix_fingerprints[name], f"{name}: 질문·문서 접두어를 교안 사양표의 입력 규약과 모델 카드에서 다시 확인하세요(없으면 빈 문자열)."
    if name in local_models_check:
        assert spec["max_input"] == local_models_check[name].max_seq_length, f"{name}: max_input은 모델 객체의 max_seq_length를 읽으세요."
        assert spec["price_per_million"] == 0.0, f"{name}: 로컬 모델은 API 단가가 없어 0.0입니다."
    else:
        assert fingerprint((spec["max_input"], float(spec["price_per_million"]))) == "5b9368a353", f"{name}: 최대 입력 토큰과 100만 토큰당 단가를 교안 사양표에서 다시 확인하세요."
print("사양표 확인 완료")
display(pd.DataFrame(model_specs).T)

## 3. 3-small 두 설정으로 임베딩하세요

같은 3-small을 1536차원과 512차원으로 받아야, 차원을 3분의 1로 줄였을 때 잃는 문항을 같은 질문으로 비교할 수 있습니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `normalize(vectors)` | 행마다 길이를 1로 맞춘 float32 배열을 반환합니다. |
| `doc_vectors_by_model`, `question_vectors_by_model` | 후보 이름을 키로 문서 벡터(394행)와 질문 벡터(24행)를 담을 빈 dict |
| `doc_seconds` | 후보 이름을 키로 문서 394편을 처리하는 데 걸린 초를 담을 빈 dict. 과제 02가 로컬 모델의 운영 비용을 계산할 때 씁니다. |

**작성:** 두 dict의 `"3-small (1536)"`, `"3-small (512)"` 값과 `doc_seconds`의 같은 두 키

**목적:** 교안 01의 OpenAI 임베딩과 `dimensions` 인자를 의약품 안내문에 적용합니다.

**구현할 처리**
1. `OpenAIEmbeddings`로 모델 객체를 만듭니다. 모델 이름은 `text-embedding-3-small`이고, `check_embedding_ctx_length=False`(긴 글을 자동으로 나눠 평균 내는 길이 검사를 끔)와 `chunk_size=100`을 넘깁니다. 512 설정은 `dimensions`로 차원을 정합니다.
2. `chunk_size=100`은 한 요청에 보낼 글의 수입니다. 안내문 394편이 약 40만 토큰이라 한 번에 보내면 요청당 30만 토큰 한도를 넘어 API 에러가 나므로 100편씩 나눠 보냅니다.
3. 문서(`drug_texts`)와 질문(`eval_texts`)을 모두 모델 객체의 `embed_documents`로 임베딩하고 `normalize`로 길이를 1로 맞춥니다. 3-small은 사양표대로 접두어가 없어 글을 그대로 넣습니다.
4. 문서 임베딩 앞뒤로 `time.perf_counter()`를 호출해 걸린 초를 `doc_seconds`에 넣습니다.

**반환값:** 문서 벡터 `(394, 차원)`, 질문 벡터 `(24, 차원)` 배열이 두 dict에 하나씩, `doc_seconds`에 초 두 개

In [ ]:
def normalize(vectors):
    """행마다 길이를 1로 맞춘 벡터 배열을 반환합니다."""
    vectors = np.asarray(vectors, dtype=np.float32)  # 임베딩 API가 돌려준 리스트를 float32 배열로
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)  # 길이가 1이면 내적이 곧 코사인 유사도


# 후보 이름을 키로 결과 모으기 (뒤의 비교표가 읽음)
doc_vectors_by_model = {}        # 문서 394편의 벡터
question_vectors_by_model = {}   # 평가 질문 24개의 벡터
doc_seconds = {}                 # 문서 394편을 처리하는 데 걸린 초

In [ ]:
# [작성] 3-small을 1536차원과 512차원으로 받아 문서와 질문을 임베딩하세요.
from langchain_openai import OpenAIEmbeddings

# 1. 1536차원(기본)으로 받는 모델을 만드세요. 문서를 100편씩 나눠 보내게 합니다.
small_1536_model = ...

# 2. 문서를 임베딩해 길이를 1로 맞추고 걸린 초를 측정하세요. 질문도 임베딩해 길이를 1로 맞추세요.
start = time.perf_counter()
doc_vectors_by_model["3-small (1536)"] = ...
doc_seconds["3-small (1536)"] = time.perf_counter() - start
question_vectors_by_model["3-small (1536)"] = ...

# 3. 512차원으로 받는 모델을 만드세요. 문서는 같은 이유로 100편씩 나눠 보냅니다.
small_512_model = ...

# 4. 2번처럼 문서·질문을 임베딩하고 걸린 초를 측정하세요.
...

In [ ]:
# [자가채점] 벡터 모양·길이를 사양표 차원과 대조합니다. API는 다시 부르지 않습니다.
for name in ["3-small (1536)", "3-small (512)"]:
    assert name in doc_vectors_by_model and name in question_vectors_by_model, f"{name}: 두 dict에 '{name}' 키로 벡터를 넣으세요."
    doc_matrix = np.asarray(doc_vectors_by_model[name])
    question_matrix = np.asarray(question_vectors_by_model[name])
    expected_dimension = model_specs[name]["dimension"]
    assert doc_matrix.shape == (len(drug_texts), expected_dimension), f"{name}: 문서 벡터 모양이 {doc_matrix.shape}입니다. (안내문 수, 사양표 차원)이어야 합니다. 512 설정은 dimensions 인자를 확인하세요."
    assert question_matrix.shape == (len(eval_texts), expected_dimension), f"{name}: 질문 벡터 모양이 {question_matrix.shape}입니다. 평가 질문(eval_texts)만 임베딩하세요."
    # 길이 1이어야 내적이 코사인 유사도
    assert np.allclose(np.linalg.norm(doc_matrix, axis=1), 1, atol=1e-3), f"{name}: 문서 벡터를 normalize로 길이 1로 맞추세요."
    assert np.allclose(np.linalg.norm(question_matrix, axis=1), 1, atol=1e-3), f"{name}: 질문 벡터를 normalize로 길이 1로 맞추세요."
    assert doc_seconds.get(name, 0) > 0, f"{name}: 문서 임베딩에 걸린 초를 doc_seconds에 넣으세요."
print("3-small 임베딩 확인 완료 | 걸린 초:", {name: round(doc_seconds[name], 1) for name in doc_seconds})

## 4. 로컬 모델 둘로 임베딩하세요

로컬 모델은 모델 카드의 입력 규약을 지켜야 제 성능이 나오므로, 사양표의 접두어를 붙여 넣습니다.

**작성:** 두 dict의 `"me5-small (384)"`, `"KURE-v1 (1024)"` 값과 `doc_seconds`의 같은 두 키

**목적:** 교안 01의 로컬 모델 임베딩을 사양표로 일반화해, 접두어를 코드에 적지 않고 사양표에서 읽게 합니다.

**구현할 처리**
1. 골격의 `local_models`에서 후보 이름과 모델 객체를 하나씩 꺼내고, 후보마다 사양표 행을 읽습니다.
2. 안내문마다 앞에 사양표의 `doc_prefix`를 붙여 모델 객체의 `encode`로 임베딩합니다. `normalize_embeddings=True`로 길이를 1로 맞추고 `show_progress_bar=False`로 진행 막대를 끕니다.
3. 문서 임베딩 앞뒤로 `time.perf_counter()`를 호출해 걸린 초를 `doc_seconds`에 넣습니다.
4. 질문마다 앞에 `query_prefix`를 붙여 같은 방법으로 임베딩합니다.

**반환값:** 문서 벡터 `(394, 차원)`, 질문 벡터 `(24, 차원)` 배열이 두 dict에 하나씩, `doc_seconds`에 초 두 개

In [ ]:
# [작성] 로컬 모델 둘로 문서와 질문을 임베딩하세요. 접두어는 사양표에서 읽습니다.
# 1. 후보 이름과 모델 객체를 짝지은 dict
local_models = {"me5-small (384)": me5_model, "KURE-v1 (1024)": kure_model}

for name, model in local_models.items():
    # 2~3. 문서마다 사양표의 doc_prefix를 붙여 임베딩하고 걸린 초를 doc_seconds에 넣으세요.
    ...

    # 4. 질문마다 사양표의 query_prefix를 붙여 임베딩하세요.
    ...

In [ ]:
# [자가채점] 모양·길이와 입력 규약을 확인합니다. 문서·질문 하나씩만 사양표 접두어로 다시 임베딩해 대조합니다.
# 대조할 문서: 가장 짧은 안내문 (접두어가 차지하는 몫이 커서 빠졌을 때 차이가 잘 드러남)
shortest = int(np.argmin([len(text) for text in drug_texts]))
local_models_check = {"me5-small (384)": me5_model, "KURE-v1 (1024)": kure_model}
for name, model in local_models_check.items():
    spec = model_specs[name]
    assert name in doc_vectors_by_model and name in question_vectors_by_model, f"{name}: 두 dict에 '{name}' 키로 벡터를 넣으세요."
    doc_matrix = np.asarray(doc_vectors_by_model[name])
    question_matrix = np.asarray(question_vectors_by_model[name])
    assert doc_matrix.shape == (len(drug_texts), spec["dimension"]), f"{name}: 문서 벡터 모양이 {doc_matrix.shape}입니다. 안내문 전체(drug_texts)를 임베딩하세요."
    assert question_matrix.shape == (len(eval_texts), spec["dimension"]), f"{name}: 질문 벡터 모양이 {question_matrix.shape}입니다. 평가 질문(eval_texts)만 임베딩하세요."
    assert np.allclose(np.linalg.norm(doc_matrix, axis=1), 1, atol=1e-3), f"{name}: normalize_embeddings=True로 길이를 1로 맞추세요."
    # 접두어를 붙여 다시 만든 벡터와 거의 같아야 함 (접두어가 빠지면 코사인이 눈에 띄게 낮아짐)
    expected_doc = model.encode(spec["doc_prefix"] + drug_texts[shortest], normalize_embeddings=True, show_progress_bar=False)
    expected_question = model.encode(spec["query_prefix"] + eval_texts[0], normalize_embeddings=True, show_progress_bar=False)
    assert float(doc_matrix[shortest] @ expected_doc) > 0.9995, f"{name}: 문서 앞에 사양표의 doc_prefix를 붙였는지, 문서 순서를 바꾸지 않았는지 확인하세요."
    assert float(question_matrix[0] @ expected_question) > 0.9995, f"{name}: 질문 앞에 사양표의 query_prefix를 붙였는지 확인하세요."
    assert doc_seconds.get(name, 0) > 0, f"{name}: 문서 임베딩에 걸린 초를 doc_seconds에 넣으세요."
print("로컬 모델 임베딩 확인 완료 | 걸린 초:", {name: round(doc_seconds[name], 1) for name in doc_seconds})

## 5. 모델 비교표를 만드세요

같은 질문 24개로 BM25 기준선과 후보 넷의 정확도를 측정하고, 선정 기준에 쓸 잘림 비율과 차원을 한 행에 모읍니다. 아래 제공 셀 셋을 먼저 실행하세요.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `bm25_top_ids` | BM25 기준선(교안 01의 `kiwi_tokenize`)이 질문마다 찾은 상위 10개 ID. 색인 시간은 `doc_seconds["BM25"]` |
| `search_top_ids(question_vectors, doc_vectors, doc_ids, top_k=10)` | 질문마다 내적이 큰 문서 `top_k`개의 ID 목록(모든 문서와 비교하는 정확 검색) |
| `evaluate_model(name, top_ids)` | 비교표 한 행(dict). 열은 `model`, `맞힌 문항`, `mean_recall_at_5`, `mrr`, `묘사 앞`, `묘사 뒤`, `이름`이고 마지막 셋은 그 묶음에서 맞힌 문항 수. 교안_01의 `evaluate_model`과 같은 지표를 계산하되, 묘사 질문을 앞·뒤로 나눠 맞힌 문항을 따로 셉니다(잘림 영향을 보려고). |
| `token_counts`, `truncated_ratio(name, token_counts)` | 후보별 문서 토큰 수, 사양표 최대 입력보다 긴 문서의 비율(BM25는 0.0) |

**작성:** 비교표 `compare_rows`

**목적:** 교안 01 4-3의 비교표처럼 후보마다 한 행을 만들어, 선정 기준을 적용할 표를 만듭니다.

**구현할 처리**
1. BM25 행: `evaluate_model`에 이름 `"BM25"`와 `bm25_top_ids`를 넘겨 행을 만들고, `truncated_ratio` 열(BM25의 잘림 비율)과 `dimension` 열(`0`, 벡터가 없음)을 붙입니다.
2. 사양표의 후보마다 질문 벡터와 문서 벡터로 정확 검색한 결과를 `evaluate_model`에 넘겨 행을 만듭니다.
3. 행에 `truncated_ratio` 열(그 후보의 잘림 비율)과 `dimension` 열(사양표의 차원)을 붙여 `compare_rows`에 더합니다.

**반환값:** `compare_rows`(dict 5개 리스트). 순서는 BM25 행, 그다음 사양표 순서의 후보 넷

In [ ]:
from kiwipiepy import Kiwi
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

kiwi = Kiwi()  # BM25 토큰화에 쓸 한국어 형태소 분석기


def kiwi_tokenize(text):
    """문서와 질문에서 명사·외국어·숫자를 같은 기준으로 추출합니다."""
    return [token.form.lower() for token in kiwi.tokenize(text.replace("･", "·"))  # 반각 가운뎃점(･)은 ·로 맞춤
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]  # N*: 명사, SL: 외국어, SN: 숫자


drug_documents = [Document(id=doc_id, page_content=text, metadata={"source_id": doc_id}) for doc_id, text in zip(drug_ids, drug_texts)]  # 원문 ID를 source_id에 담음

# BM25 색인과 걸린 시간 (k=10: MRR을 10위까지 보므로 상위 10개를 돌려받음)
start = time.perf_counter()
bm25 = BM25Retriever.from_documents(drug_documents, preprocess_func=kiwi_tokenize, k=10)
doc_seconds["BM25"] = time.perf_counter() - start

# 평가 질문마다 BM25 상위 10개의 원문 ID
bm25_top_ids = []
for question in eval_texts:
    bm25_top_ids.append([doc.metadata["source_id"] for doc in bm25.invoke(question)])
print("BM25 색인 문서:", len(drug_documents), "| 색인 시간(초):", round(doc_seconds["BM25"], 3))

In [ ]:
def search_top_ids(question_vectors, doc_vectors, doc_ids, top_k=10):
    """질문마다 내적이 큰 문서 top_k개의 원문 ID를 순위대로 담은 목록을 반환합니다."""
    scores = question_vectors @ doc_vectors.T  # (질문 수, 문서 수) 점수표, 모든 문서와 비교하는 정확 검색

    # 질문마다 점수가 큰 문서 top_k개를 원문 ID로
    top_ids = []
    for question_scores in scores:
        order = np.argsort(-question_scores, kind="stable")[:top_k]  # 동점이면 문서 순서 유지(stable)
        top_ids.append([doc_ids[position] for position in order])
    return top_ids


# 검색 품질 평가 단원의 함수 (predicted는 검색이 낸 문서 id 목록(순위순), relevant는 정답 id 목록)
def recall_at_k(predicted, relevant, k):
    """전체 관련 문서 중 상위 k개가 찾아낸 비율(관련 수 / 전체 관련 수)."""
    # 분모는 그 질문의 정답 수 (정답 하나면 0 또는 1, 둘이면 0, 0.5, 1)
    hits = sum(1 for p in predicted[:k] if p in relevant)
    return hits / len(relevant)


def mrr(predicted, relevant):
    """첫 번째 관련 문서 순위의 역수. 목록 안에 없으면 0."""
    # 검색 결과 전체(상위 10개)에서 첫 정답 찾기 (순위는 1부터)
    for i, p in enumerate(predicted, 1):
        if p in relevant:
            return 1 / i
    return 0.0


def evaluate_model(name, top_ids):
    """평가셋 질문별 검색 결과를 정답과 비교해 Recall@5·MRR·맞힌 문항과 묶음별 맞힌 문항을 담은 한 행(dict)을 반환합니다."""
    recalls = []
    reciprocal_ranks = []
    found = 0
    # 묶음별 맞힌 문항: 묘사는 근거 위치(앞·뒤)로 나눠 잘림의 영향을 따로 봄
    group_found = {"묘사 앞": 0, "묘사 뒤": 0, "이름": 0}
    for question, retrieved_ids in zip(eval_questions, top_ids, strict=True):
        recall = recall_at_k(retrieved_ids, question["relevant_doc_ids"], 5)
        recalls.append(recall)
        reciprocal_ranks.append(mrr(retrieved_ids, question["relevant_doc_ids"]))  # 상위 10개 안 첫 정답의 순위로
        if recall > 0:  # 정답 문서가 하나라도 상위 5개 안이면 맞힌 문항
            found += 1
            if question["question_type"] == "이름":
                group_found["이름"] += 1
            else:
                group_found["묘사 " + question["section"]] += 1

    # 평균을 소수점 셋째 자리로 반올림해 비교표 한 행으로
    return {
        "model": name,
        "맞힌 문항": found,
        "mean_recall_at_5": round(float(np.mean(recalls)), 3),
        "mrr": round(float(np.mean(reciprocal_ranks)), 3),
        "묘사 앞": group_found["묘사 앞"],
        "묘사 뒤": group_found["묘사 뒤"],
        "이름": group_found["이름"],
    }

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("text-embedding-3-small")  # API가 요금을 매기는 토큰 수와 같은 기준


def count_tokens(texts):
    """후보 모델별로 문서마다의 토큰 수 목록을 담은 dict를 반환합니다."""
    openai_counts = [len(encoding.encode(text)) for text in texts]
    # 로컬 모델은 자기 토크나이저로 실제 입력(사양표의 문서 접두어 포함)을 자르지 않고 셈 (verbose=False는 길이 경고 끄기)
    me5_counts = [len(me5_model.tokenizer(model_specs["me5-small (384)"]["doc_prefix"] + text, verbose=False)["input_ids"]) for text in texts]
    kure_counts = [len(kure_model.tokenizer(model_specs["KURE-v1 (1024)"]["doc_prefix"] + text, verbose=False)["input_ids"]) for text in texts]

    return {"3-small (1536)": openai_counts, "3-small (512)": openai_counts,
            "me5-small (384)": me5_counts, "KURE-v1 (1024)": kure_counts}  # 3-small 두 설정은 같은 토크나이저


def truncated_ratio(name, token_counts):
    """후보 name에서 최대 입력보다 길어 뒷부분이 잘리는 문서의 비율을 반환합니다."""
    if name not in model_specs:  # BM25는 최대 입력이 없어 잘림 없음
        return 0.0

    # 사양표의 최대 입력보다 긴 문서 수의 비율
    over = 0
    for count in token_counts[name]:
        if count > model_specs[name]["max_input"]:
            over += 1
    return round(over / len(token_counts[name]), 3)


token_counts = count_tokens(drug_texts)  # 안내문의 후보별 토큰 수

# 후보마다 평균·최대 토큰을 최대 입력과 견준 길이 표
length_rows = []
for name in model_specs:
    length_rows.append({"model": name, "평균 토큰": round(float(np.mean(token_counts[name])), 1),
                        "최대 토큰": int(np.max(token_counts[name])), "최대 입력": model_specs[name]["max_input"],
                        "truncated_ratio": truncated_ratio(name, token_counts)})
display(pd.DataFrame(length_rows))

In [ ]:
# [작성] BM25 기준선과 후보 넷의 비교표를 만드세요.
compare_rows = []

# 1. BM25 기준선 행 (벡터가 없어 차원 0)
...

for name in model_specs:
    # 2. 질문 벡터와 문서 벡터로 정확 검색하고 한 행을 평가하세요.
    ...

    # 3. 잘림 비율과 차원을 붙여 compare_rows에 더하세요.
    ...

display(pd.DataFrame(compare_rows))

In [ ]:
# [자가채점] 행 수·순서·열과 잘림·차원 값을 확인하고, 마지막 행은 검색을 다시 해 대조합니다. 어느 행이 이기는지는 보지 않습니다.
expected_columns = ["model", "맞힌 문항", "mean_recall_at_5", "mrr", "묘사 앞", "묘사 뒤", "이름", "truncated_ratio", "dimension"]
assert isinstance(compare_rows, list) and len(compare_rows) == 5, "compare_rows에 BM25 1행과 후보 넷, 모두 5행을 담으세요."
assert [row["model"] for row in compare_rows] == ["BM25"] + list(model_specs), "행 순서는 BM25, 그다음 사양표 순서의 후보 넷입니다."
for row in compare_rows:
    assert isinstance(row, dict) and list(row) == expected_columns, f"행의 열은 {expected_columns} 순서입니다. evaluate_model 결과에 truncated_ratio, dimension을 차례로 붙이세요."
    assert 0 <= row["맞힌 문항"] <= len(eval_questions), "맞힌 문항은 0부터 평가 문항 수 사이입니다."
    assert row["묘사 앞"] + row["묘사 뒤"] + row["이름"] == row["맞힌 문항"], "evaluate_model이 만든 행을 그대로 쓰세요."
    assert row["truncated_ratio"] == truncated_ratio(row["model"], token_counts), f"{row['model']}: truncated_ratio(name, token_counts)를 붙이세요."
    expected_dimension = model_specs[row["model"]]["dimension"] if row["model"] in model_specs else 0
    assert row["dimension"] == expected_dimension, f"{row['model']}: dimension은 사양표의 차원(BM25는 0)입니다."
# 마지막 행(KURE-v1)을 같은 모델의 질문·문서 벡터로 다시 검색해 대조
check_name = compare_rows[4]["model"]
check_ids = search_top_ids(question_vectors_by_model[check_name], doc_vectors_by_model[check_name], drug_ids)
assert evaluate_model(check_name, check_ids)["맞힌 문항"] == compare_rows[4]["맞힌 문항"], f"{check_name}: 그 후보의 질문 벡터와 문서 벡터로 검색한 결과를 평가하세요."
print("비교표 확인 완료:", len(compare_rows), "행")

## 6. 선정 기준을 함수로 옮기세요

같은 기준을 다른 평가셋에도 다시 적용하려면, 머리말의 우선순위 0~3단계를 순서 그대로 함수로 옮겨 둡니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `best_within(rows, column, tolerance)` | `column` 값이 가장 큰 행과 차이가 `tolerance` 이하인 행만 원래 순서대로 담은 목록 |
| `smallest(rows, column)` | `column` 값이 가장 작은 행(dict). 같으면 앞의 행 |

**작성:** `select_drug_model(rows)` 함수

**목적:** 교안 01의 `select_model`처럼 단계마다 남은 후보를 출력하며 거르되, 과제의 기준과 순서(잘림 -> 정확도 -> 차원)를 따릅니다.

**구현할 처리**
1. 0단계: `model`이 `"BM25"`인 행을 뺍니다.
2. 1단계: `truncated_ratio`가 0인 행만 남깁니다.
3. 2단계: `맞힌 문항`이 가장 많은 행과 1문항 이내 차이인 행만 남깁니다.
4. 3단계: 남은 행 가운데 `dimension`이 가장 작은 행을 고릅니다.
5. 각 단계 뒤에 단계 이름과 남은 행의 `model` 목록을 한 줄씩 출력합니다(네 줄).

**반환값:** 고른 행(입력 `rows`에 있던 dict 그대로)

In [ ]:
def best_within(rows, column, tolerance):
    """column 값이 가장 큰 행과 차이가 tolerance 이하인 행만 원래 순서대로 담은 목록을 반환합니다."""
    # 가장 큰 값 찾기
    best = rows[0][column]
    for row in rows:
        if row[column] > best:
            best = row[column]

    # 가장 큰 값과 tolerance 이내인 행만 남김
    kept = []
    for row in rows:
        if best - row[column] <= tolerance:
            kept.append(row)
    return kept


def smallest(rows, column):
    """column 값이 가장 작은 행을 반환합니다. 같으면 앞의 행."""
    choice = rows[0]
    for row in rows[1:]:
        if row[column] < choice[column]:
            choice = row
    return choice

In [ ]:
# [작성] 선정 기준 0~3단계를 차례로 적용해 고른 행을 반환하는 함수를 완성하세요.
def select_drug_model(rows):
    """선정 기준 0~3단계를 차례로 적용해 고른 행(dict)을 반환합니다. 단계마다 남은 후보를 출력합니다."""
    # 0단계: BM25는 기준선이라 고르지 않음
    ...

    # 1단계: 잘림 비율이 0인 행만
    ...

    # 2단계: 맞힌 문항이 가장 많은 행과 1문항 이내인 행만
    ...

    # 3단계: 남은 행 가운데 차원이 가장 작은 행
    ...

In [ ]:
# [자가채점] 가짜 고정 표 네 개로 단계의 순서와 경계를 확인합니다. 실제 비교표는 쓰지 않습니다.
import contextlib
import io


def fake_row(model, found, truncated, dimension):
    """자가채점용 가짜 비교표 한 행을 만듭니다(비교표와 같은 열)."""
    return {"model": model, "맞힌 문항": found, "mean_recall_at_5": round(found / 24, 3), "mrr": 0.5,
            "묘사 앞": 0, "묘사 뒤": 0, "이름": found, "truncated_ratio": truncated, "dimension": dimension}


# (표 이름, 행 목록, 골라야 할 행 위치, 틀렸을 때 볼 규칙)
fake_tables = [
    ("가 정확도 1위가 조금 잘리는 표", [fake_row("모델 가", 20, 0.005, 384), fake_row("모델 나", 15, 0.0, 1024)], 1,
     "잘림 비율이 0이 아닌 행은 조금만 잘려도, 정확도가 높아도 먼저 빼야 합니다(1단계를 정확도보다 먼저)."),
    ("나 1문항 차이인 두 행", [fake_row("모델 가", 15, 0.0, 1024), fake_row("모델 나", 14, 0.0, 512)], 1,
     "1문항 이내는 같은 수준이고, 같은 수준이면 차원이 작은 행을 고릅니다."),
    ("다 1문항 차이로 앞선 BM25", [fake_row("BM25", 16, 0.0, 0), fake_row("모델 가", 15, 0.0, 1024)], 1,
     "BM25는 기준선이라 고르지 않습니다(0단계)."),
    ("라 2문항 차이인 두 행", [fake_row("모델 가", 16, 0.0, 1024), fake_row("모델 나", 14, 0.0, 512)], 0,
     "같은 수준은 1문항 이내까지입니다. 2문항 차이는 같은 수준이 아닙니다."),
]
for table_name, rows, expected_position, rule in fake_tables:
    printed = io.StringIO()
    try:
        # 가짜 표에서는 단계별 출력을 모아 줄 수만 봄
        with contextlib.redirect_stdout(printed):
            chosen = select_drug_model(rows)
    except Exception as error:
        raise AssertionError(f"[{table_name}] 실행 중 에러 {error!r}. 앞 단계가 남긴 행만 다음 단계에 넘기는지, 단계 순서가 지문과 같은지 확인하세요.") from error
    assert isinstance(chosen, dict), "고른 행(dict)을 return 하세요. `...`를 채웠는지도 확인하세요."
    assert chosen is rows[expected_position], f"[{table_name}] {rule}"
    assert len(printed.getvalue().strip().splitlines()) >= 4, "단계마다 남은 후보를 한 줄씩(네 줄) 출력하세요."
print("선정 함수 확인 완료: 가짜 표", len(fake_tables), "개 통과")

## 7. 결정과 후보를 파일로 남기세요

과제 02는 이 파일의 결정으로 저장 형식과 저장소를 정하고, 로컬 모델의 색인 시간으로 비용 축을 계산하므로, 결정과 후보별 값을 함께 남깁니다.

**작성:** 고른 행 `choice`, 결과 dict `record`, 파일 `output/gwaje01_embedding_choice.json`

**목적:** 교안 01 따라하기의 결과 파일처럼, 뒤 과제가 읽을 결정과 후보표를 한 파일에 남깁니다.

**구현할 처리**
1. 실제 비교표 `compare_rows`에 `select_drug_model`을 적용한 결과를 `choice`에 담습니다.
2. `record`는 `model`(`choice`의 값)과 `candidates`(빈 리스트로 시작) 두 키의 dict입니다.
3. `compare_rows`의 행마다 후보 dict 하나를 `candidates`에 더합니다. 후보는 다섯 개가 됩니다.
4. 후보 dict의 키는 `model`, `dimension`, `mean_recall_at_5`, `doc_embed_seconds`, `truncated_ratio`, `external_api` 여섯 개입니다. 값은 아래 표를 따릅니다.
5. `save_json`으로 `record`를 `gwaje01_embedding_choice.json`에 저장합니다.

| 키 | 값 |
|---|---|
| `model`, `dimension`, `mean_recall_at_5`, `truncated_ratio` | 그 행의 값 |
| `doc_embed_seconds` | `doc_seconds`의 그 모델 값을 소수 셋째 자리로 반올림 |
| `external_api` | 사양표의 값. BM25는 사양표에 없어 `False` |

**반환값:** `choice`(고른 행), `record`(두 키의 dict), 저장한 JSON 파일

In [ ]:
# [작성] 비교표에 선정 기준을 적용하고 결정과 후보를 파일로 남기세요.
# 1. 실제 비교표에 선정 함수 적용
choice = ...

# 2. 고른 모델과 빈 후보 목록
record = ...

# 3~4. 비교표의 행마다 후보 dict를 만들어 record["candidates"]에 더하세요.
...

# 5. 파일로 저장
...

In [ ]:
# [자가채점] 저장한 파일을 다시 읽어 결정·후보 다섯 개와 그 값이 비교표·사양표와 같은지 확인합니다.
assert isinstance(choice, dict) and choice in compare_rows, "choice에 select_drug_model(compare_rows)가 고른 행을 담으세요."
assert choice["model"] != "BM25", "BM25는 기준선이라 고를 수 없습니다. 선정 함수를 확인하세요."
saved_path = output_dir / "gwaje01_embedding_choice.json"
assert saved_path.exists(), "save_json으로 output/gwaje01_embedding_choice.json을 저장하세요."
saved = json.loads(saved_path.read_text(encoding="utf-8"))
assert list(saved) == ["model", "candidates"], "record의 키는 model, candidates 순서입니다."
assert saved["model"] == choice["model"], "model은 choice의 값을 넣으세요."
candidate_keys = ["model", "dimension", "mean_recall_at_5", "doc_embed_seconds", "truncated_ratio", "external_api"]
assert len(saved["candidates"]) == len(compare_rows), "후보는 비교표의 행마다 하나, 다섯 개입니다."
for candidate, row in zip(saved["candidates"], compare_rows):
    name = row["model"]
    # 과제 02는 키 이름으로 읽으므로 순서는 보지 않음
    assert set(candidate) == set(candidate_keys), f"후보 dict에는 {candidate_keys} 여섯 키가 있어야 합니다."
    assert candidate["model"] == name, "후보는 비교표의 행 순서대로 옮기세요."
    assert candidate["dimension"] == row["dimension"] and candidate["mean_recall_at_5"] == row["mean_recall_at_5"] and candidate["truncated_ratio"] == row["truncated_ratio"], f"{name}: dimension, mean_recall_at_5, truncated_ratio는 그 행의 값입니다."
    assert candidate["doc_embed_seconds"] == round(doc_seconds[name], 3), f"{name}: doc_embed_seconds는 doc_seconds를 소수 셋째 자리로 반올림한 값입니다."
    expected_api = model_specs[name]["external_api"] if name in model_specs else False
    assert candidate["external_api"] is expected_api, f"{name}: external_api는 사양표 값(BM25는 False)입니다."
print("결과 파일 확인 완료:", saved_path.name, "| 결정:", saved["model"], "| 후보:", len(saved["candidates"]))

## 결정 기록 ADR-001을 쓰세요

**작성:** 아래 마크다운 셀을 더블클릭해 괄호 안 안내를 지우고 제목과 네 칸(상태·맥락·결정·결과)을 채웁니다. 제목은 고른 기술이 드러나게 씁니다(예: "ADR-00N: ○○로 △△ 사용"). 맥락은 정해야 할 문제·제약·후보를 2~3문장으로 쓰고 결정은 쓰지 않습니다. 결정은 "…를 쓴다."로 시작하고, 아래 `근거:`에 검토한 선택지마다 한 줄씩 버린 이유를 쓴 뒤 고른 것이 이긴 이유를 씁니다. 결과 아래에는 `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`(어떤 가정이나 수치가 바뀌면 이 결정을 다시 내려야 하는지: 바뀌는 것 -> 다시 할 일)을 씁니다. ADR은 서비스 문서처럼 교안·과제·절 번호 없이 사실을 직접 씁니다. 숫자는 자신이 실행한 비교표와 단계별 출력에서 옮기세요.

**목적:** 비교표의 숫자로 선정 기준의 각 단계가 무엇을 지웠는지 설명하고, 결론이 바뀌는 조건을 남깁니다.

**설명할 내용**
1. 상태: 제안·채택·거절·대체됨 가운데 하나(이 결정을 받아들였으면 `채택`)
2. 맥락: 정해야 할 것(임베딩 모델), 서비스 제약(잘림·외부 API), 검토한 후보와 비교 방법(같은 수준으로 보는 폭)
3. 결정: 고른 모델을 "…를 쓴다."로 쓰고, `근거:`에 후보마다 한 줄씩 비교표 숫자로 씁니다.
    - 빠진 모델마다 이유(잘림 비율, 맞힌 문항 차이)
    - me5-small과 KURE-v1의 묘사 `앞`과 `뒤`의 맞힌 문항을 함께 보고, me5-small이 덜 맞힌 것을 잘림 탓으로 볼 수 있는지(잘리지 않는 `앞`에서도 차이가 나는지)
    - 3-small (1536)과 (512)의 차이(차원을 줄였을 때 잃은 문항)
    - BM25 기준선과의 차이
    - 고른 모델이 이긴 이유
4. 결과: `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`을 둘 이상(평가셋, 문서 길이, 검색 전략 등)

#### ADR-001: (고른 기술이 드러나는 제목)

- **상태**: (제안·채택·거절·대체됨 가운데 하나)
- **맥락**: (2~3문장: 정할 문제(임베딩 모델), 지켜야 할 조건(잘림·외부 API), 검토한 후보(모델 넷과 기준선). 결정은 쓰지 않음)
- **결정**: (고른 것을 한 문장으로: …를 쓴다)
    - 근거: (후보마다 한 줄: 뺀 이유(잘림, 맞힌 문항, me5-small의 묘사 앞·뒤, 3-small 두 차원, BM25와의 차이). 마지막 줄은 고른 것이 이긴 이유)
- **결과**
    - 얻는 것: (이 결정으로 얻는 것)
    - 감수하는 것: (대신 감수하는 것)
    - 다시 검토할 조건: (바뀌는 가정이나 수치 -> 다시 할 일)